# HBO Trip Generation Model Development

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from sklearn.preprocessing import StandardScaler

from anfis_toolbox import ANFISRegressor

# R2 score for a zero-intercept
def r2_zero_intercept(y_true, y_pred):
    # Sum of squared residuals
    ss_res = np.sum((y_true - y_pred) ** 2)
    # Uncentered total sum of squares (sum of squares of y_true)
    ss_tot_uncentered = np.sum(y_true ** 2)
    return 1 - (ss_res / ss_tot_uncentered)

### 1. Loading Dataset

In [2]:
# 1. LOAD DATASET
df = pd.read_csv("data/salfit_trip_data.csv")

# 2. DEFINE INPUTS / OUTPUT
X = df[['SIZE', 'DRV', 'HHTYP']]
y = df['HBO']

# 3. TRAIN / TEST SPLIT (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# modified split to match the paper
df_train = pd.read_csv('data/salfit_trip_data_train.csv')
X_train = df_train[['SIZE', 'DRV', 'HHTYP']].values
y_train = df_train['HBO'].values
print(X_train.shape, y_train.shape)

df_test = pd.read_csv('data/salfit_trip_data_test.csv')
X_test = df_test[['SIZE', 'DRV', 'HHTYP']].values
y_test = df_test['HBO'].values
print(X_test.shape, y_test.shape)

# 4. SCALE INPUTS
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


(256, 3) (256,)
(53, 3) (53,)


### 2. Model Development Using Multiple Linear Regression (MLR)


In [3]:
mlr = LinearRegression(fit_intercept=False)

mlr.fit(X_train, y_train)

# Predictions
y_train_pred_mlr = mlr.predict(X_train)
y_test_pred_mlr = mlr.predict(X_test)

# Metrics
rmse_train_mlr = np.sqrt(mean_squared_error(y_train, y_train_pred_mlr))
rmse_test_mlr = np.sqrt(mean_squared_error(y_test, y_test_pred_mlr))

mae_train_mlr = mean_absolute_error(y_train, y_train_pred_mlr)
mae_test_mlr = mean_absolute_error(y_test, y_test_pred_mlr)

r2_train_mlr = r2_zero_intercept(y_train, y_train_pred_mlr)
r2_test_mlr = r2_zero_intercept(y_test, y_test_pred_mlr)

# Results
print("\nRegression Equation:")
print(
    f"HBALL = {mlr.intercept_:.3f} "
    f"+ ({mlr.coef_[0]:.3f} * SIZE) "
    f"+ ({mlr.coef_[1]:.3f} * DRV) "
    f"+ ({mlr.coef_[2]:.3f} * HHTYP)"
)

print("\n MLR RESULTS")

print(f"Train RMSE : {rmse_train_mlr:.4f}")
print(f"Test RMSE  : {rmse_test_mlr:.4f}")

print(f"Train MAE  : {mae_train_mlr:.4f}")
print(f"Test MAE   : {mae_test_mlr:.4f}")

print(f"Train R²   : {r2_train_mlr:.4f}")
print(f"Test R²    : {r2_test_mlr:.4f}")



Regression Equation:
HBALL = 0.000 + (0.415 * SIZE) + (0.289 * DRV) + (1.218 * HHTYP)

 MLR RESULTS
Train RMSE : 1.5778
Test RMSE  : 1.9647
Train MAE  : 1.2825
Test MAE   : 1.5141
Train R²   : 0.8065
Test R²    : 0.7878


### 3. Model Development Using ANFIS 

In [4]:
max_epochs = 100

train_rmse_history = []
test_rmse_history = []

best_test_rmse = np.inf
best_epoch = 0
best_model = None

print("\nTraining ANFIS...\n")

for epoch in range(1, max_epochs + 1):

    # CREATE MODEL
    model = ANFISRegressor(
        n_mfs=3,
        mf_type='gaussian',
        optimizer='hybrid',
        epochs=epoch,
        verbose=False
    )

    # TRAIN
    model.fit(X_train, y_train)

    # TRAIN PREDICTIONS
    y_train_pred = model.predict(X_train)
    train_rmse = np.sqrt(
        mean_squared_error(y_train, y_train_pred)
    )

    # TEST PREDICTIONS
    y_test_pred = model.predict(X_test)
    test_rmse = np.sqrt(
        mean_squared_error(y_test, y_test_pred)
    )

    # STORE HISTORY
    train_rmse_history.append(train_rmse)
    test_rmse_history.append(test_rmse)

    print(
        f"Epoch {epoch:3d} | "
        f"Train RMSE: {train_rmse:.4f} | "
        f"Test RMSE: {test_rmse:.4f}"
    )

    # EARLY STOPPING CONDITION
    if test_rmse < best_test_rmse:

        best_test_rmse = test_rmse
        best_epoch = epoch
        best_model = model

    else:
        print("\nValidation error started increasing.")
        print("Stopping training to avoid overfitting.")
        break

# FINAL BEST MODEL
print("\nBEST ANFIS MODEL")
print(f"Best Epoch : {best_epoch}")
print(f"Best Test RMSE : {best_test_rmse:.4f}")

# FINAL EVALUATION
y_train_pred_final = best_model.predict(X_train)
y_test_pred_final = best_model.predict(X_test)

# TRAIN RMSE, MAE, R2
rmse_train = np.sqrt(
    mean_squared_error(y_train, y_train_pred_final)
)

mae_train = mean_absolute_error(
    y_train,
    y_train_pred_final
)

r2_train = r2_zero_intercept(
    y_train,
    y_train_pred_final
)

# TEST RMSE, MAE, R2
rmse_test = np.sqrt(
    mean_squared_error(y_test, y_test_pred_final)
)

mae_test = mean_absolute_error(
    y_test,
    y_test_pred_final
)

r2_test = r2_zero_intercept(
    y_test,
    y_test_pred_final
)

# RESULTS

print("\nANFIS RESULTS")

print("\nTraining Results:")
print(f"RMSE : {rmse_train:.4f}")
print(f"MAE  : {mae_train:.4f}")
print(f"R²   : {r2_train:.4f}")

print("\nTesting Results:")
print(f"RMSE : {rmse_test:.4f}")
print(f"MAE  : {mae_test:.4f}")
print(f"R²   : {r2_test:.4f}")




Training ANFIS...

Epoch   1 | Train RMSE: 1.4267 | Test RMSE: 13.2027
Epoch   2 | Train RMSE: 1.4267 | Test RMSE: 13.2095

Validation error started increasing.
Stopping training to avoid overfitting.

BEST ANFIS MODEL
Best Epoch : 1
Best Test RMSE : 13.2027

ANFIS RESULTS

Training Results:
RMSE : 1.4267
MAE  : 1.1247
R²   : 0.8418

Testing Results:
RMSE : 13.2027
MAE  : 4.3277
R²   : -8.5835


### Performance Comparison and Validation

In [5]:
print("\n Training Results")

comparison_training = pd.DataFrame({

    'Method': ['MLR', 'ANFIS'],

    'RMSE': [
        rmse_train_mlr,
        rmse_train
    ],

    'MAE': [
        mae_train_mlr,
        mae_train
    ],

    'R2': [
        r2_train_mlr,
        r2_train
    ]
})

print(comparison_training)

print("\n Testing Results")

comparison_testing = pd.DataFrame({

    'Method': ['MLR', 'ANFIS'],

    'RMSE': [
        rmse_test_mlr,
        rmse_test
    ],

    'MAE': [
        mae_test_mlr,
        mae_test
    ]
})

print(comparison_testing)



 Training Results
  Method      RMSE       MAE        R2
0    MLR  1.577754  1.282454  0.806538
1  ANFIS  1.426650  1.124654  0.841820

 Testing Results
  Method       RMSE       MAE
0    MLR   1.964689  1.514053
1  ANFIS  13.202694  4.327748
